# LoRA Adapter + Base Model → Gradio Pricer (Inference on Modal)

Load a **base model** and **LoRA adapters** from HuggingFace (replace the dummy names below with your model IDs). **Inference runs on Modal** — the model loads and runs in Modal's GPU containers; the Gradio UI in this notebook calls the remote `price` function.

Run this notebook with: `modal run week8/lora_gradio_pricer.ipynb` so the app is registered and Gradio can call Modal.

## 1. Install dependencies

## 2. Config — replace with your HuggingFace model and adapter IDs

In [3]:
BASE_MODEL = "unsloth/Qwen2.5-Math-1.5B-bnb-4bit"     
ADAPTER_REPO = "martinsawojide/price-2026-03-06_17.59.28"
ADAPTER_REVISION = None             
GPU = "T4"                         
HF_SECRET_NAME = "huggingface-secret"   
# Pushover: create a Modal secret with PUSHOVER_TOKEN (app token) and PUSHOVER_USER (your user key).
# Leave None to disable push notifications to phone.
PUSHOVER_SECRET_NAME = "pushover-secret"

# Keep at least one container warm to avoid cold start (continuous conversation without long waits).
# Set to 0 to scale-to-zero and save cost (first request after idle will have cold start).
MIN_CONTAINERS = 0
# Seconds an idle container stays up before scaling down (when above min_containers). Default 120.
SCALEDOWN_WINDOW = 300

## 3. Modal app — model loads and inference on Modal

In [4]:
import modal
from modal import Image

app = modal.App("lora-gradio-pricer")
image = (
    Image.debian_slim()
    .pip_install("torch", "transformers", "peft", "accelerate", "bitsandbytes", "requests")
    .env({
        "BASE_MODEL": BASE_MODEL,
        "ADAPTER_REPO": ADAPTER_REPO,
        "ADAPTER_REVISION": str(ADAPTER_REVISION) if ADAPTER_REVISION else "",
    })
)
secrets = [modal.Secret.from_name(HF_SECRET_NAME)]
if PUSHOVER_SECRET_NAME:
    secrets.append(modal.Secret.from_name(PUSHOVER_SECRET_NAME))

PREFIX = "Price is $"
QUESTION = "What does this cost to the nearest dollar?"


@app.cls(
    image=image,
    secrets=secrets,
    gpu=GPU,
    timeout=600,
    min_containers=MIN_CONTAINERS,
    scaledown_window=SCALEDOWN_WINDOW,
    region="eu",
)
class Pricer:
    @modal.enter()
    def setup(self):
        import logging
        import os
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
        from peft import PeftModel

        logging.basicConfig(
            level=logging.INFO,
            format="[%(asctime)s] %(levelname)s %(name)s: %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S",
        )
        self._log = logging.getLogger("pricer")

        base_id = os.environ["BASE_MODEL"]
        adapter_id = os.environ["ADAPTER_REPO"]
        revision = os.environ.get("ADAPTER_REVISION") or None

        self._log.info("Loading tokenizer and base model: %s", base_id)
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
        self.tokenizer = AutoTokenizer.from_pretrained(base_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "right"
        base_model = AutoModelForCausalLM.from_pretrained(
            base_id, quantization_config=quant_config, device_map="auto"
        )
        self._log.info("Loading LoRA adapter: %s", adapter_id)
        self.model = PeftModel.from_pretrained(base_model, adapter_id, revision=revision)
        self.prefix = os.environ.get("PREFIX", "Price is $")
        self.question = os.environ.get("QUESTION", "What does this cost to the nearest dollar?")
        self._log.info("Setup complete — model ready")

    @modal.method()
    def price(self, description: str) -> float:
        import os
        import re
        import torch
        import requests
        from transformers import set_seed

        self._log.info("price request: %s", (description[:80] + "..." if len(description) > 80 else description))
        set_seed(42)
        prompt = f"{self.question}\n\n{description}\n\n{self.prefix}"
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=8)
        text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if self.prefix not in text:
            price = 0.0
        else:
            contents = text.split(self.prefix)[1].replace(",", "")
            match = re.search(r"[-+]?\d*\.\d+|\d+", contents)
            price = float(match.group()) if match else 0.0

        self._log.info("inference done: price=%.2f", price)

        msg_body = f"Estimated price: ${price:.2f}\n\nProduct: {description[:500]}"

        # Push notification to phone via Pushover (if secret is configured)
        token = os.environ.get("PUSHOVER_TOKEN")
        user = os.environ.get("PUSHOVER_USER")
        if token and user:
            try:
                requests.post(
                    "https://api.pushover.net/1/messages",
                    data={
                        "token": token,
                        "user": user,
                        "message": msg_body,
                        "title": "Pricer",
                    },
                    timeout=10,
                )
                self._log.info("Pushover notification sent")
            except Exception as e:
                self._log.warning("Pushover failed: %s", e)
        else:
            self._log.debug("Pushover skipped (no secret)")

        return price

## 4. Gradio UI (calls Modal for inference)

In [5]:
import gradio as gr

def ui_predict(description: str) -> str:
    if not description.strip():
        return "Enter a product description."
    # Inference runs on Modal
    price = Pricer().price.remote(description).get()
    return f"Estimated price: ${price:.2f}"

with gr.Blocks(title="Pricer — LoRA + Base (Modal)") as demo:
    gr.Markdown("## Price estimator (inference on Modal)")
    gr.Markdown("Enter a product description to get an estimated price (nearest dollar).")
    inp = gr.Textbox(
        label="Product description",
        placeholder="e.g. USB-C laptop charger 65W",
        lines=3,
    )
    out = gr.Textbox(label="Result")
    inp.submit(ui_predict, inputs=inp, outputs=out)
    gr.Button("Get price").click(ui_predict, inputs=inp, outputs=out)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/martinsawojide/Desktop/Andela_AI_Eng/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/martinsawojide/Desktop/Andela_AI_Eng/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/martinsawojide/Desktop/Andela_AI_Eng/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/martinsawojide/Desktop/Andela_AI_Eng/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^